In [1]:
try:
    %load_ext autoreload --quiet
except ImportError:
    %reload_ext autoreload
%autoreload 2

import hashlib
from typing import List, Any
from flow_merge.lib.snapshot.data_architecture._normalized_slices import NormalizedSlice
from pydantic import BaseModel, computed_field, Field
import datetime
from pathlib import Path
import json

class MergePlan(BaseModel):
    created_at: datetime.datetime = Field(default=datetime.datetime.now())
    base_model: str
    tokenizer_mode: str
    tokenizer_interpolation_method: str
    slices: List[NormalizedSlice]
    lib_version: str

    @classmethod
    def from_config(cls, config: Any) -> "MergePlan":
        return cls(
            created_at=datetime.datetime.now(),
            base_model=config.base_model,
            tokenizer_mode=config.tokenizer_mode,
            tokenizer_interpolation_method=config.tokenizer_interpolation_method,
            slices=config.slices,
            lib_version=config.lib_version,
        )

    @classmethod
    def from_file(cls, file_path: Path | str) -> "MergePlan":
        with open(Path(file_path).resolve(), "rb") as f:
            parsed = json.load(f)
            return cls(**parsed)

    @computed_field
    @property
    def sha(self) -> str:
        obj = self.model_dump_json(exclude={"created_at", "sha"}).encode("utf-8")
        return hashlib.md5(obj).hexdigest()

mp = MergePlan.from_file("../flow_merge/lib/example-slices.json")


In [2]:
print(mp.sha)

dacbc64dcbfa795dc7eb9a2c267bf051


In [5]:
from flow_merge.lib.model.architecture import ModelWeightArch
from flow_merge.lib.tensor.loader import TensorRepository
from toolz import compose
from flow_merge.lib.merge_methods import method_classes, method_configs, MergeMethodIdentifier
from typing import Tuple, Dict
from flow_merge.lib.model import Model
from flow_merge.lib.tokenizer import get_merge_tokenizer, Tokenizer

# get models from all the slices
all_model_path_occurrences: List[str|None] = [model_field.model for x in mp.slices for model_field in x.sources]
all_distinct_model_paths: List[str|None] = list({model for model in all_model_path_occurrences})
all_model_objects: List[Dict[str, Model]] = [{model_path: Model.from_path(model_path)} for model_path in all_distinct_model_paths]

# what is the 'mode' for base model - which model is most frequently the base model
all_models: List[Tuple[str,bool]] = [(model_field.model, model_field.is_base) for x in mp.slices for model_field in x.sources]
the_most_frequent_base_model_name = max(set(name for name, is_base in all_models if is_base), 
                                   key=lambda name: sum(is_base for n, is_base in all_models if n == name), 
                                   default=None)
base_model = next((model for model in all_model_objects.values() if model.id == the_most_frequent_base_model_name), None)

# construct merged tokenizer out of the distinct models
common_tokenizer = get_merge_tokenizer(models=all_model_objects.values(), base_model=base_model, tokenizer_mode=mp.tokenizer_mode)


class Stage(BaseModel):
    models: List[Model]
    base_model: Model
    tokenizer: Tokenizer
    # ...

executable_slices: List[NormalizedSlice] = mp.slices
    
Stage(
    models=all_model_objects, 
    base_model=base_model, 
    tokenizer=common_tokenizer,
    slices=executable_slices
    # ...
)


class StageService:
    def __init__(self, models, base_model, tokenizer, slices):
        self.models = models
        self.base_model = base_model
        self.tokenizer = tokenizer
        self.slices = slices

    # transform slices
    @staticmethod
    def _add_tensors(slice_obj):
        for src in slice_obj.sources:
            src.tensor = TensorRepository.get_tensor(
                    shards=src.model.shards,
                    tensor_key=src.model.architecture.get_weight(src.layer).name,
                )
        return slice_obj
    
    @staticmethod 
    def _add_merge_method(self, slice_obj):
        method_settings = method_configs[slice_obj.merge_method.name]
        slice_obj.merge_method = {"name": slice_obj.merge_method.name,
                                  "method": method_classes[slice_obj.merge_method.name],
                                  "settings": method_settings(**slice_obj.merge_method.params)}
        return slice_obj
    
    # THIRD STATIC METHOD?
    # def _get_merged_config(self):
    # merged_model_config = base_model.architecture.config
    # merged_model_config.num_hidden_layers = num_hidden_layers
    
    @staticmethod
    def exec_slice(slice_obj):
        slice_obj.merge_method["method"].merge(args)

    @staticmethod
    def run(self):
        for executable_slice in self.slices:
            self.exec_slice(executable_slice)

    # EXCEPTION CASE - INTERPOLATION
    def create_executable_interpolation_slice(self, slice_obj):
        if slice_obj.merge_method.name == MergeMethodIdentifier.INTERPOLATE:
            slice_obj.merge_method = {"name": slice_obj.merge_method.name,
                                      "method": None,
                                      "settings": None}

            if self.tokenizer.input_ids_mappings:
                merged_model_config.vocab_size = len(
                    self.tokenizer.tokenizer.get_vocab()
                )

            all_tensors = \
                { base_model: base_model_tensor, **models_tensors }
            hidden_dim = _validate_tensor_shapes(
                base_model_weight=, # ModelWeight
                tensors=all_tensors, # Dict[Model, torch, Tensor]
                base_model_layer_type=task_base_model_weight.layer_type # str
            )
        return slice_obj

    # TAKE THIS ELSEWHERE / REWRITE
    @staticmethod
    def _validate_tensor_shapes(
            base_model_weight: ModelWeight,
            tensors: Dict[Model, torch.Tensor],
            base_model_layer_type: str
    ):
        hidden_size = next(iter(tensors.values())).shape[
            1 if base_model_layer_type == "embedding" else 0
        ]
        for model, tensor in tensors.items():
            current_size = tensor.shape[1] if base_model_layer_type == "embedding" else tensor.shape[0]
            if current_size != hidden_size:
                raise RuntimeError(
                    f"Tensor shape mismatch in '{base_model_weight.name}'. Expected {hidden_size}, but {model.path} has {current_size}."
                )
        return hidden_size



# RUN CALLS ?

# InterpolationRunner.interpolate(
#     base_model=base_model,
#     all_tensors=all_tensors,
#     merge_method=method_config,
#     input_ids_mappings=tokenizer.input_ids_mappings,
#     sources=sources,
#     hidden_dim=hidden_dim
# )
        
# method_config.method.merge(
#         weight=task_base_model_weight,
#         base_model_tensor=base_model_tensor,
#         models_tensors=models_tensors,
#         merge_method_settings=method_config.settings,
#         base_model=base_model
#     )


# EXAMPLE FROM MINAAM
# def _old_merge_sources(self):
#     for slice in self..normalized:
# 
#         base_model_weight = None
#         models_with_weights = {}
#         for source in slice["sources"]:
#             # if merge_method is passthrough, pass it along to the merged model
#             if source["base_model"]:
#                 base_model_weight = self.enriched_snapshot.base_model.architecture.get_weight(source["layer"])
#                 continue
# 
#             path_or_id = source["model"]
#             model = self._get_model_by_id(path_or_id)
# 
#             model_weight = model.architecture.get_weight(source["layer"])
# 
#             # using the model as key to get the weight
#             models_with_weights[model] = model_weight
# 
#         self.merger.merge(
#             base_model=self.enriched_snapshot.base_model, # Model
#             task_base_model_weight=base_model_weight, # ModelWeight
#             task_models_with_weights=models_with_weights, # Dict[Model, ModelWeight]
#             tokenizer=self.enriched_snapshot.tokenizer, # Tokenizer
#             method_config=method_config, # Any
#             sources=slice["sources"] # Any
#         )
        


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 97)